# Demo + Deploy (model already on HF — no training)

Two things only: (1) the base-vs-tuned **money shot** for your video, and (2)
deploy a **free CPU Hugging Face Space**. Run top to bottom. Needs an HF token.

## 1. Install + get the repo helpers

In [ ]:
!pip install -q transformers torch gradio huggingface_hub peft
import os
if not os.path.isdir("QuestionGen"):
    !git clone -q https://github.com/gabriel-xiong/QuestionGen.git
%cd QuestionGen

## 2. Config — SET THESE
Point at your hosted model. If you pushed a **merged** model set `IS_MERGED=True`;
if you pushed only the **LoRA adapter**, set `IS_MERGED=False`.

In [ ]:
HF_USER  = "gabriel-xiong"
MODEL_ID = "gabriel-xiong/apbio-item-generator-qwen3-1.7b-lora"  # your HF adapter repo
IS_MERGED = False                                            # this repo is a LoRA adapter
BASE_ID  = "Qwen/Qwen3-1.7B"
SPACE_ID = f"{HF_USER}/apbio-item-generator"                 # Space to create

## 3. Log in to Hugging Face

In [ ]:
from huggingface_hub import login
login()  # paste your hf_... token

## 4. Video money shot — base vs tuned on the SAME (unseen) prompt

In [ ]:
import json, sys, torch; sys.path.insert(0,'scripts')
import gen_spec
from transformers import AutoModelForCausalLM, AutoTokenizer

def load(model_id, merged=True):
    if merged:
        tok = AutoTokenizer.from_pretrained(model_id)
        m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype='auto', device_map='auto')
    else:
        from peft import PeftModel
        tok = AutoTokenizer.from_pretrained(BASE_ID)
        m = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype='auto', device_map='auto')
        m = PeftModel.from_pretrained(m, model_id)
    return m.eval(), tok

def gen(m, tok, msgs):
    x = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    i = tok(x, return_tensors='pt').to(m.device)
    o = m.generate(**i, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][i['input_ids'].shape[1]:], skip_special_tokens=True)

sc = json.loads(open('data/eval_scenarios_ood.jsonl').readline())   # unseen topic
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
msgs = [{'role':'system','content':system},{'role':'user','content':user}]

bm, bt = load(BASE_ID, merged=True)
print('=== BASE Qwen3-1.7B ===
', gen(bm, bt, msgs))
tm, tt = load(MODEL_ID, merged=IS_MERGED)
print('
=== TUNED (yours) ===
', gen(tm, tt, msgs))

## 5. Deploy the free CPU Space
Uploads the Gradio app pointed at your model. (The Space loads with plain
transformers, so it needs a **merged** model — if you only have an adapter,
push a merged version first or the Space will fail to load.)

In [ ]:
import re, shutil, os
src = open('scripts/app.py', encoding='utf-8').read()
src = re.sub(r'MODEL_ID = "[^"]*"', f'MODEL_ID = "{MODEL_ID}"', src, count=1)
open('app_space.py','w',encoding='utf-8').write(src)
open('requirements.txt','w').write('transformers
torch
gradio
peft
')

from huggingface_hub import HfApi, create_repo
create_repo(SPACE_ID, repo_type='space', space_sdk='gradio', exist_ok=True)
api = HfApi()
api.upload_file(path_or_fileobj='app_space.py', path_in_repo='app.py', repo_id=SPACE_ID, repo_type='space')
api.upload_file(path_or_fileobj='requirements.txt', path_in_repo='requirements.txt', repo_id=SPACE_ID, repo_type='space')
api.upload_file(path_or_fileobj='data/apbio_misconceptions.json', path_in_repo='data/apbio_misconceptions.json', repo_id=SPACE_ID, repo_type='space')
print('Space building: https://huggingface.co/spaces/' + SPACE_ID)
print('First build takes a few minutes; the URL is permanent and free.')

## Done
Space URL is permanent (free CPU: ~10-30s/gen, sleeps when idle, wakes on visit).
Switch hardware to ZeroGPU/GPU in Space Settings for faster inference.